# Defend against attrition: the full analysis in one notebook

This notebook is the complete attrition workstream for our Lloyds project, with all of the code and explanations in one place.

**The question:** can free, public data tell the bank which business customers might be about to leave, so it can act early? We use public data only. No Lloyds internal data is used.

**Attrition is three different things, and they do not look the same in the data:**

- **Dormancy:** a company goes quiet and stops trading.
- **Closure:** a company is struck off, goes insolvent, or is dissolved.
- **Bank switch:** a healthy company moves its borrowing to a rival bank.

The first two are signs of distress. The third is a healthy customer leaving for a competitor. Because the clues are different, we look at them separately.

**Data sources (both public and free):** Companies House (the official record of every UK company, plus its live service for loans and filings) and GDELT (a free news index).

The notebook has two parts. First, the **engine**: the functions that do the work, each explained in plain English. Then the **analysis**: loading the data, building the signals, and the findings.

### Before we start: load the API key

The live Companies House lookups need a free API key, kept in a `.env` file so it never goes into the notebook or the repo. This cell loads it.

In [1]:
# Load the Companies House API key from the project's .env file (one level up).
# This keeps the key out of the notebook and out of version control.
import os
from pathlib import Path

_envp = Path("../.env")
if _envp.exists():
    for _line in _envp.read_text(encoding="utf-8").splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

print("Companies House API key loaded:", "CH_API_KEY" in os.environ)

Companies House API key loaded: True


## Part 1: The engine

You do not need to read the code to follow the story. Each block below has a plain explanation first, then the code that does it.

### 1a. Turning raw records into simple signals

Companies House fields are messy. This block converts them into plain signals:
- is the company healthy or heading for trouble (from its status),
- is it dormant (from its accounts type),
- how big is it, roughly (a size band from its accounts type),
- what sector is it in (worked out from its industry code),
- is it filing its accounts late (a known early warning sign).

In [1]:
# === Engine: turn raw Companies House fields into simple signals ===
from __future__ import annotations

from datetime import date, datetime
from typing import Iterable, Optional

# ---------------------------------------------------------------------------
# 1. Company status -> attrition state (the "closure" axis)
# ---------------------------------------------------------------------------
# CompanyStatus is a free text field in the bulk file. We collapse it into a
# small set of attrition states. Distress states are ordered worst-last so a
# single severity can be derived if needed.

HEALTHY = "healthy_active"
STRIKE_OFF = "strike_off_risk"
INSOLVENCY = "insolvency"
DISSOLVED = "dissolved"
STATUS_OTHER = "other"

# Lower-cased exact matches taken from the live status vocabulary.
_INSOLVENCY_STATUSES = {
    "liquidation",
    "in administration",
    "in administration/administrative receiver",
    "administration order",
    "voluntary arrangement",
    "live but receiver manager on at least one charge",
    "receivership",
}


def classify_status(company_status: Optional[str]) -> str:
    """Map a raw CompanyStatus string to an attrition state.

    Unknown or missing values map to STATUS_OTHER rather than guessing.
    """
    if company_status is None:
        return STATUS_OTHER
    s = company_status.strip().lower()
    if s == "":
        return STATUS_OTHER
    if s == "active":
        return HEALTHY
    if "proposal to strike off" in s or "strike-off" in s or "strike off" in s:
        return STRIKE_OFF
    if "dissolved" in s:
        return DISSOLVED
    if s in _INSOLVENCY_STATUSES or "administration" in s or "liquidation" in s:
        return INSOLVENCY
    return STATUS_OTHER


def is_distress_status(company_status: Optional[str]) -> bool:
    """True if the status is any non-healthy, non-other attrition state."""
    return classify_status(company_status) in {STRIKE_OFF, INSOLVENCY, DISSOLVED}


# ---------------------------------------------------------------------------
# 2. Accounts category -> dormancy flag and size band (the "dormancy" axis)
# ---------------------------------------------------------------------------
# Account category is a coarse proxy. Post 6 April 2025 thresholds:
#   micro:  turnover < 1m,  small: < 15m,  medium: < 54m.
# Lloyds BCB segments by turnover: BB < 3m, SME 3-25m, Midcorp 25-500m.
# The mapping below is intentionally conservative and documented as a proxy.

DORMANT = "dormant"
NO_ACCOUNTS = "no_accounts"
TRADING = "trading"

_DORMANT_CATEGORIES = {"dormant"}
_NO_ACCOUNTS_CATEGORIES = {"no accounts filed", "accounts type not available"}


def classify_accounts_activity(account_category: Optional[str]) -> str:
    """Return TRADING, DORMANT, or NO_ACCOUNTS from Accounts.AccountCategory."""
    if account_category is None:
        return NO_ACCOUNTS
    c = account_category.strip().lower()
    if c == "":
        return NO_ACCOUNTS
    if c in _DORMANT_CATEGORIES:
        return DORMANT
    if c in _NO_ACCOUNTS_CATEGORIES:
        return NO_ACCOUNTS
    return TRADING


# Size band proxy. Categories that imply a larger filer map upward. Categories
# that carry no size information (dormant, no accounts, subsidiary exemptions)
# map to UNKNOWN so they are not misread as small.
BB = "BB"            # micro / very small, broadly Business Banking (< ~3m)
SME = "SME"          # small filers (3-25m bracket, approximate)
MID = "Midcorp"      # medium accounts (25-500m bracket, approximate)
LARGE = "Large"      # full / group filers, often above the BCB sweet spot
SIZE_UNKNOWN = "unknown"

_SIZE_MAP = {
    "micro entity": BB,
    "total exemption small": BB,
    "small": SME,
    "total exemption full": SME,
    "unaudited abridged": SME,
    "audited abridged": SME,
    "medium": MID,
    "full": LARGE,
    "group": LARGE,
}


def size_band(account_category: Optional[str]) -> str:
    """Map account category to an approximate Lloyds size band proxy."""
    if account_category is None:
        return SIZE_UNKNOWN
    return _SIZE_MAP.get(account_category.strip().lower(), SIZE_UNKNOWN)


# ---------------------------------------------------------------------------
# 3. SIC code -> UK SIC section -> Lloyds target sector
# ---------------------------------------------------------------------------
# Bulk file SIC fields look like "62020 - Information technology consultancy".
# We work from the 5 digit code, take the 2 digit division, then the section.

import re

_SIC_CODE_RE = re.compile(r"(\d{5})")


def extract_sic_code(sic_text: Optional[str]) -> Optional[str]:
    """Pull the 5 digit SIC code out of a bulk-file SIC string."""
    if not sic_text:
        return None
    m = _SIC_CODE_RE.search(sic_text)
    return m.group(1) if m else None


def sic_division(sic_code: Optional[str]) -> Optional[int]:
    """First two digits of a 5 digit SIC code, as an int."""
    if not sic_code or len(sic_code) < 2 or not sic_code[:2].isdigit():
        return None
    return int(sic_code[:2])


# UK SIC 2007 section letter by division range.
_SECTION_RANGES = [
    ("A", 1, 3), ("B", 5, 9), ("C", 10, 33), ("D", 35, 35), ("E", 36, 39),
    ("F", 41, 43), ("G", 45, 47), ("H", 49, 53), ("I", 55, 56), ("J", 58, 63),
    ("K", 64, 66), ("L", 68, 68), ("M", 69, 75), ("N", 77, 82), ("O", 84, 84),
    ("P", 85, 85), ("Q", 86, 88), ("R", 90, 93), ("S", 94, 96), ("T", 97, 98),
    ("U", 99, 99),
]


def sic_section(sic_code: Optional[str]) -> Optional[str]:
    """Map a 5 digit SIC code to its UK SIC 2007 section letter."""
    div = sic_division(sic_code)
    if div is None:
        return None
    for letter, lo, hi in _SECTION_RANGES:
        if lo <= div <= hi:
            return letter
    return None


# Lloyds target sectors (the eight from the brief), plus OTHER.
SECTOR_MANUFACTURING = "Manufacturing"
SECTOR_PUBLIC = "Public sector, education & charities"
SECTOR_HEALTHCARE = "Healthcare"
SECTOR_TECH_PROF = "Technology, legal & professional"
SECTOR_AGRICULTURE = "Agriculture"
SECTOR_REAL_ESTATE = "Real estate"
SECTOR_WHOLESALE_RETAIL = "Wholesale & retail"
SECTOR_FAST_GROWTH = "Fast growth & emerging"
SECTOR_OTHER = "Other"

# Section letter -> target sector for the straightforward cases.
_SECTION_TO_SECTOR = {
    "C": SECTOR_MANUFACTURING,
    "A": SECTOR_AGRICULTURE,
    "G": SECTOR_WHOLESALE_RETAIL,
    "L": SECTOR_REAL_ESTATE,
    "Q": SECTOR_HEALTHCARE,
    "O": SECTOR_PUBLIC,
    "P": SECTOR_PUBLIC,
    "J": SECTOR_TECH_PROF,
    "M": SECTOR_TECH_PROF,
}

# Fast growth and emerging: explicit 5 digit codes (software, data, biotech, fintech
# adjacent). Takes priority over the section mapping, per the team's draft.
_FAST_GROWTH_CODES = {
    "62011", "62012", "62020", "62030", "62090",
    "63110", "63120",
    "72110", "72190", "72200",
    "66190",
}

# Company categories that signal a charity or social enterprise regardless of SIC.
_CHARITY_CATEGORIES = {
    "community interest company",
    "charitable incorporated organisation",
    "scottish charitable incorporated organisation",
    "registered society",
}


def target_sector(
    sic_texts: Iterable[Optional[str]],
    company_category: Optional[str] = None,
) -> str:
    """Assign one Lloyds target sector to a company.

    Rules (documented in ATTRITION_WORKSTREAM.md):
      1. If any SIC code is in the fast-growth list, label Fast growth & emerging.
      2. Else, if the company type is a charity/social enterprise, label Public.
      3. Else, use the section of the first SIC code that maps to a target sector,
         scanning the provided SIC codes in order (primary SIC first).
      4. Else, Other.
    """
    codes = [extract_sic_code(t) for t in sic_texts]
    codes = [c for c in codes if c]

    # Rule 1: fast-growth override.
    if any(c in _FAST_GROWTH_CODES for c in codes):
        return SECTOR_FAST_GROWTH

    # Rule 2: charity / social enterprise by company type.
    if company_category and company_category.strip().lower() in _CHARITY_CATEGORIES:
        return SECTOR_PUBLIC

    # Rule 3: first mappable section wins (primary SIC first).
    for c in codes:
        sector = _SECTION_TO_SECTOR.get(sic_section(c))
        if sector:
            return sector

    return SECTOR_OTHER


# ---------------------------------------------------------------------------
# 4. Filing punctuality (a leading distress signal for dormancy and closure)
# ---------------------------------------------------------------------------

def _coerce_date(value) -> Optional[date]:
    if value is None:
        return None
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    if isinstance(value, str):
        v = value.strip()
        if v == "":
            return None
        for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
            try:
                return datetime.strptime(v, fmt).date()
            except ValueError:
                continue
        return None
    return None


def days_overdue(next_due, as_of) -> Optional[int]:
    """Days a filing is overdue as of a reference date.

    Positive means overdue, zero or negative means not yet due. Returns None if
    either date is missing or unparseable.
    """
    due = _coerce_date(next_due)
    ref = _coerce_date(as_of)
    if due is None or ref is None:
        return None
    return (ref - due).days


def is_overdue(next_due, as_of, grace_days: int = 0) -> Optional[bool]:
    """True if a filing is overdue by more than grace_days as of a date."""
    d = days_overdue(next_due, as_of)
    if d is None:
        return None
    return d > grace_days


### 1b. Reading a company's loans and filings

The downloaded file only says how many loans a company has, not who lent the money. This block talks to the live Companies House service to get the detail:
- the **loans (charges)**, including the **lender's name** and the dates, which lets us see if a company moved from one bank to another,
- the **filing history**, which lets us date the first sign of trouble (a strike-off notice, an insolvency filing, or a switch to dormant accounts).

It also tidies messy lender names (so all the HSBC entities count as HSBC) and ignores security agents and trustees, which are not real banks.

In [1]:
# === Engine: read company loans and filing history from Companies House ===
from __future__ import annotations

import datetime as _dt
import json
import os
import re
import time
from pathlib import Path
from typing import Optional

BASE_URL = "https://api.company-information.service.gov.uk"

# 600 requests / 300 seconds = 0.5s minimum spacing. Add headroom.
_MIN_INTERVAL_SECONDS = 0.6


# ---------------------------------------------------------------------------
# API key loading
# ---------------------------------------------------------------------------
def load_api_key(env_path: Optional[str] = None) -> str:
    """Load CH_API_KEY from the environment or a .env file.

    Raises a clear error if the key is missing, so set-up problems are obvious.
    """
    key = os.getenv("CH_API_KEY")
    if not key:
        # Try python-dotenv if available, then fall back to a tiny parser.
        try:
            from dotenv import load_dotenv  # type: ignore

            load_dotenv(env_path)
            key = os.getenv("CH_API_KEY")
        except Exception:
            key = _read_env_file(env_path)
    if not key:
        raise RuntimeError(
            "CH_API_KEY not found. Create a .env file with "
            "CH_API_KEY=your_key (see the docstring for where to get one)."
        )
    return key


def _read_env_file(env_path: Optional[str]) -> Optional[str]:
    path = Path(env_path) if env_path else Path(".env")
    if not path.exists():
        return None
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith("CH_API_KEY="):
            return line.split("=", 1)[1].strip().strip('"').strip("'")
    return None


# ---------------------------------------------------------------------------
# Client
# ---------------------------------------------------------------------------
class CompaniesHouseClient:
    """Thin, polite client with on-disk caching and rate limiting."""

    def __init__(
        self,
        api_key: Optional[str] = None,
        cache_dir: str = "data/raw/api_cache",
        min_interval: float = _MIN_INTERVAL_SECONDS,
    ):
        self.api_key = api_key or load_api_key()
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.min_interval = min_interval
        self._last_call = 0.0
        self._session = None  # lazy, so importing this module needs no requests

    def _get_session(self):
        if self._session is None:
            import requests  # lazy import

            s = requests.Session()
            s.auth = (self.api_key, "")
            self._session = s
        return self._session

    def _throttle(self):
        elapsed = time.monotonic() - self._last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self._last_call = time.monotonic()

    def _cache_path(self, key: str) -> Path:
        # Keep only filename-safe characters (Windows forbids ? & = % etc).
        safe = re.sub(r"[^A-Za-z0-9._-]", "_", key).strip("_")
        return self.cache_dir / f"{safe}.json"

    def _get(self, path: str, use_cache: bool = True) -> Optional[dict]:
        """GET an API path, with caching. Returns parsed JSON or None on 404."""
        cache_file = self._cache_path(path)
        if use_cache and cache_file.exists():
            return json.loads(cache_file.read_text(encoding="utf-8"))

        session = self._get_session()
        url = f"{BASE_URL}{path}"
        for attempt in range(5):
            self._throttle()
            resp = session.get(url, timeout=30)
            if resp.status_code == 200:
                data = resp.json()
                if use_cache:
                    cache_file.write_text(json.dumps(data), encoding="utf-8")
                return data
            if resp.status_code == 404:
                return None
            if resp.status_code == 429:
                # Rate limited: back off and retry.
                time.sleep(2 ** attempt)
                continue
            resp.raise_for_status()
        raise RuntimeError(f"Giving up on {url} after repeated rate limiting")

    # --- endpoints -------------------------------------------------------
    def search_companies(self, query: str, items: int = 1) -> list:
        """Search companies by name. Returns the raw 'items' list (top matches)."""
        from urllib.parse import quote

        data = self._get(
            f"/search/companies?q={quote(query)}&items_per_page={items}",
            use_cache=False,
        )
        return data.get("items", []) if data else []

    def get_company(self, company_number: str) -> Optional[dict]:
        return self._get(f"/company/{company_number}")

    def get_charges(self, company_number: str) -> Optional[dict]:
        return self._get(f"/company/{company_number}/charges")

    def get_filing_history(self, company_number: str) -> Optional[dict]:
        return self._get(f"/company/{company_number}/filing-history")

    def get_officers(self, company_number: str) -> Optional[dict]:
        return self._get(f"/company/{company_number}/officers")


# ---------------------------------------------------------------------------
# Pure analysis helpers (no network) - unit tested
# ---------------------------------------------------------------------------
def parse_charges(charges_response: Optional[dict]) -> list:
    """Flatten a /charges response into simple per-charge dicts.

    Each item: {created_on, satisfied_on, status, classification, lenders}.
    """
    if not charges_response:
        return []
    out = []
    for item in charges_response.get("items", []):
        lenders = [
            p.get("name", "").strip()
            for p in item.get("persons_entitled", [])
            if p.get("name")
        ]
        classification = ""
        cls = item.get("classification")
        if isinstance(cls, dict):
            classification = cls.get("description", "")
        out.append(
            {
                "created_on": item.get("created_on"),
                "satisfied_on": item.get("satisfied_on"),
                "status": item.get("status"),  # outstanding / satisfied / part-satisfied
                "classification": classification,
                "lenders": lenders,
            }
        )
    return out


def lender_timeline(charges: list) -> list:
    """Charges sorted by creation date, each with its lenders and active window.

    Returns a list of (created_on, satisfied_on, lender, status) tuples, one per
    lender per charge, ordered by created_on. Useful for spotting a switch.
    """
    rows = []
    for c in charges:
        for lender in c["lenders"] or [""]:
            rows.append(
                (c.get("created_on") or "", c.get("satisfied_on"), lender, c.get("status"))
            )
    rows.sort(key=lambda r: r[0])
    return rows


# Charge status vocabulary used by the API. A charge still owed is "outstanding"
# or "part-satisfied"; a cleared charge is "satisfied" or "fully-satisfied".
OUTSTANDING_STATUSES = {"outstanding", "part-satisfied"}
SATISFIED_STATUSES = {"satisfied", "fully-satisfied"}


def is_outstanding(charge: dict) -> bool:
    return (charge.get("status") or "").lower() in OUTSTANDING_STATUSES


def is_satisfied(charge: dict) -> bool:
    return (charge.get("status") or "").lower() in SATISFIED_STATUSES


# Major UK banking groups: map any subsidiary name containing a keyword to the
# group, so 'Hsbc Equipment Finance' and 'Hsbc UK Bank' both become HSBC. This is
# a coarse entity resolution on the lender side; it reduces false switch signals
# between subsidiaries of the same group.
_BANK_GROUPS = {
    "LLOYDS": ("lloyds", "bank of scotland", "halifax", "hbos", "black horse", "mbna"),
    "HSBC": ("hsbc", "midland bank"),
    "BARCLAYS": ("barclays",),
    "NATWEST": (
        "natwest", "national westminster", "royal bank of scotland", "rbs",
        "ulster bank", "coutts", "lombard north",
    ),
    "SANTANDER": ("santander", "abbey national"),
    "NATIONWIDE": ("nationwide",),
    "VIRGIN MONEY": ("virgin money", "clydesdale", "yorkshire bank", "cybg"),
    "TSB": ("tsb",),
    "METRO BANK": ("metro bank",),
    "CO-OPERATIVE BANK": ("co-operative bank", "co-op bank"),
    "ALDERMORE": ("aldermore",),
    "SHAWBROOK": ("shawbrook",),
    "CLOSE BROTHERS": ("close brothers",),
}


def _normalise_lender(name: str) -> str:
    """Normalise a lender name to a banking group where possible.

    Falls back to a suffix-stripped upper-case name for non-bank or unknown
    lenders, so 'Barclays Bank PLC' ~ 'BARCLAYS BANK PLC.'.
    """
    low = name.lower()
    for group, keywords in _BANK_GROUPS.items():
        if any(k in low for k in keywords):
            return group
    n = name.upper().strip().rstrip(".")
    for suffix in (" PLC", " LIMITED", " LTD", " LLP"):
        if n.endswith(suffix):
            n = n[: -len(suffix)].strip()
    return n


# Markers that a "person entitled" is a security trustee or agent acting for a
# syndicate or bondholders, not the lender itself. These dominate large/complex
# debt and must be stripped before reading a bank-supplier change.
_AGENT_MARKERS = (
    "security agent", "security trustee", "as trustee", "as security",
    "nominee", "fiduciary", "trust corporation", "trustees limited",
    "trustee company", "as agent",
)

LENDER_BANK = "bank"
LENDER_AGENT = "security_agent"
LENDER_OTHER = "other"  # non-bank lender, PE/credit fund, landlord, individual


def lender_type(name: str) -> str:
    """Classify a charge holder as a bank, a security agent, or other."""
    low = name.lower()
    for keywords in _BANK_GROUPS.values():
        if any(k in low for k in keywords):
            return LENDER_BANK
    if any(m in low for m in _AGENT_MARKERS):
        return LENDER_AGENT
    return LENDER_OTHER


def _bank_set(charges: list, status_pred) -> set:
    """Recognised banking groups among lenders matching a status predicate."""
    out = set()
    for c in charges:
        if not status_pred(c):
            continue
        for l in c["lenders"]:
            if l and lender_type(l) == LENDER_BANK:
                out.add(_normalise_lender(l))
    return out


def detect_bank_switch(charges: list) -> dict:
    """Bank-supplier change restricted to recognised banks.

    Ignores security agents, PE/credit funds, landlords, and individuals, so the
    signal reflects a genuine move between high-street/commercial banks. A bank
    is "lost" if its charge is cleared and it holds no outstanding charge, and a
    different bank is "gained" on an outstanding charge.
    """
    current = _bank_set(charges, is_outstanding)
    past = _bank_set(charges, is_satisfied)
    lost = past - current
    gained = current - past
    return {
        # Clean A -> B switch: a bank dropped and a different bank picked up.
        "bank_switch": bool(lost) and bool(gained),
        # More common and arguably stronger attrition signal: had bank charges,
        # now holds no outstanding bank charge at all.
        "lost_all_banks": bool(past) and not current,
        # Reduced the number of bank relationships (consolidation).
        "reduced_banks": len(lost) > 0,
        "banks_lost": sorted(lost),
        "banks_gained": sorted(gained),
        "current_banks": sorted(current),
        "past_banks": sorted(past),
        "n_current_banks": len(current),
        "n_past_banks": len(past),
    }


def _parse_iso(value):
    """Parse a 'YYYY-MM-DD' string to a date, or return None."""
    if not value:
        return None
    try:
        return _dt.date.fromisoformat(value[:10])
    except (ValueError, TypeError):
        return None


def _months_between(later, earlier) -> "float | None":
    """Approximate months between two dates (later minus earlier)."""
    a, b = _parse_iso(later) if isinstance(later, str) else later, \
        _parse_iso(earlier) if isinstance(earlier, str) else earlier
    if a is None or b is None:
        return None
    return (a - b).days / 30.44


def bank_loss_date(charges: list):
    """Date the company most recently lost a bank (latest satisfied bank charge
    whose bank is no longer among its outstanding banks). Returns ISO str or None.
    """
    current = current_lenders_banks(charges)
    dates = []
    for c in charges:
        if not is_satisfied(c):
            continue
        for l in c["lenders"]:
            if l and lender_type(l) == LENDER_BANK:
                grp = _normalise_lender(l)
                if grp not in current and c.get("satisfied_on"):
                    dates.append(c["satisfied_on"])
    return max(dates) if dates else None


def lost_bank_within(charges: list, ref_date: str, months: int = 24) -> bool:
    """True if a bank was lost in the window of `months` ending at ref_date.

    A bank is "lost" when its charge is satisfied and that bank holds no current
    outstanding charge. We then check the satisfied date falls inside the window
    just before ref_date. ref_date is an ISO string (for example an event date).
    This is the windowed version used for the lead-lag test.
    """
    ref = _parse_iso(ref_date)
    if ref is None:
        return False
    current = current_lenders_banks(charges)
    window_days = months * 30.44
    for c in charges:
        if not is_satisfied(c):
            continue
        sat = _parse_iso(c.get("satisfied_on"))
        if sat is None:
            continue
        for l in c["lenders"]:
            if l and lender_type(l) == LENDER_BANK and _normalise_lender(l) not in current:
                gap_days = (ref - sat).days
                if 0 <= gap_days <= window_days:
                    return True
    return False


def recent_bank_loss(charges: list, as_of: str, months: int = 24) -> bool:
    """True if the company has lost all its banks and the loss is recent.

    "Recent" means the latest lost-bank charge was satisfied within `months` of
    the reference date. A fresh loss is a sharper attrition signal than an old one.
    """
    bs = detect_bank_switch(charges)
    if not bs["lost_all_banks"]:
        return False
    loss = bank_loss_date(charges)
    gap = _months_between(as_of, loss)
    return gap is not None and 0 <= gap <= months


def current_lenders_banks(charges: list) -> set:
    """Banks (grouped) on charges still owed. Bank-only version of current_lenders."""
    return _bank_set(charges, is_outstanding)


def current_lenders(charges: list) -> set:
    """Distinct lenders (grouped) on charges still owed."""
    return {
        _normalise_lender(l)
        for c in charges
        if is_outstanding(c)
        for l in c["lenders"]
        if l
    }


def past_lenders(charges: list) -> set:
    """Distinct lenders (grouped) whose charges are cleared (paid off)."""
    return {
        _normalise_lender(l)
        for c in charges
        if is_satisfied(c)
        for l in c["lenders"]
        if l
    }


def detect_switch(charges: list) -> dict:
    """Heuristic for a bank supplier change from a single charges pull.

    Signal: at least one lender's charge has been satisfied (a credit
    relationship ended) and a different lender now holds an outstanding charge
    (a new relationship began). Both being true is consistent with a switch of
    secured lender between banks.

    Returns a dict with the boolean flag and the lender sets, so the caller can
    inspect direction. This only sees secured lending, not current accounts.
    """
    current = current_lenders(charges)
    past = past_lenders(charges)
    gained = current - past
    lost = past - current
    switched = bool(lost) and bool(gained)
    return {
        "switched": switched,
        "lost_lenders": sorted(lost),
        "gained_lenders": sorted(gained),
        "current_lenders": sorted(current),
        "past_lenders": sorted(past),
    }


# ---------------------------------------------------------------------------
# Filing history: the time dimension for a real attrition label
# ---------------------------------------------------------------------------
# A single snapshot shows a state, not a change. The filing history endpoint
# lists a company's filings over time, so we can date when it first showed signs
# of dormancy or closure. That lets us build an attrition event timeline without
# waiting for a second snapshot.

def parse_filing_history(response: "dict | None") -> list:
    """Flatten a /filing-history response into simple per-filing dicts."""
    if not response:
        return []
    out = []
    for item in response.get("items", []):
        out.append(
            {
                "date": item.get("date"),
                "category": (item.get("category") or "").lower(),
                "type": (item.get("type") or "").upper(),
                "description": (item.get("description") or "").lower(),
            }
        )
    return out


def _is_strike_off(f: dict) -> bool:
    return (
        f["category"] == "gazette"
        or f["type"].startswith("GAZ")
        or f["type"] in {"DS01", "DS02"}
        or "strike" in f["description"]
        or "gazette" in f["description"]
    )


def _is_insolvency(f: dict) -> bool:
    return (
        f["category"] == "insolvency"
        or any(
            w in f["description"]
            for w in ("insolvency", "liquidation", "administration", "receiver", "winding")
        )
    )


def _is_dormant_accounts(f: dict) -> bool:
    return f["category"] == "accounts" and "dormant" in f["description"]


def _earliest_date(filings: list, predicate):
    dates = [f["date"] for f in filings if f.get("date") and predicate(f)]
    return min(dates) if dates else None


def extract_attrition_events(filings: list) -> dict:
    """Find the first date the company shows each kind of distress in its filings.

    Returns the first date of a strike-off step, an insolvency filing, and a
    dormant accounts filing, plus the earliest of the three and which it was.
    Any field is None if that event never appears.
    """
    strike = _earliest_date(filings, _is_strike_off)
    insolv = _earliest_date(filings, _is_insolvency)
    dormant = _earliest_date(filings, _is_dormant_accounts)

    candidates = {
        "strike_off": strike,
        "insolvency": insolv,
        "dormant_accounts": dormant,
    }
    present = {k: v for k, v in candidates.items() if v}
    if present:
        earliest_type = min(present, key=present.get)
        earliest_date = present[earliest_type]
    else:
        earliest_type = None
        earliest_date = None

    return {
        "first_strike_off": strike,
        "first_insolvency": insolv,
        "first_dormant_accounts": dormant,
        "first_event_type": earliest_type,
        "first_event_date": earliest_date,
        "has_event": earliest_date is not None,
    }


### 1c. Looking up news coverage

This block asks GDELT, a free news index, how many news articles mention a company. It cleans the company name first, because the registered name is messy.

In [1]:
# === Engine: look up news coverage for a company (GDELT, free) ===
from __future__ import annotations

import json
import re
import time
from pathlib import Path

DOC_API = "https://api.gdeltproject.org/api/v2/doc/doc"

# Legal suffixes we strip so the name matches news text better.
_SUFFIXES = (
    "limited", "ltd", "plc", "llp", "llp.", "ltd.", "plc.",
    "company", "co", "uk", "holdings", "group", "the",
)


def clean_company_name(name: str) -> str:
    """Turn a registered name into a cleaner phrase for a news search.

    Drops punctuation and common legal suffixes, so 'ACME WIDGETS LIMITED'
    becomes 'acme widgets'. Returns a lower-case string.
    """
    n = (name or "").lower()
    n = re.sub(r"[^a-z0-9 ]+", " ", n)
    words = [w for w in n.split() if w and w not in _SUFFIXES]
    return " ".join(words).strip()


def parse_artlist(payload: "dict | None") -> list:
    """Flatten a GDELT artlist JSON payload into simple article dicts."""
    if not payload:
        return []
    out = []
    for a in payload.get("articles", []):
        out.append(
            {
                "title": a.get("title", ""),
                "url": a.get("url", ""),
                "domain": a.get("domain", ""),
                "seendate": a.get("seendate", ""),
                "language": a.get("language", ""),
                "country": a.get("sourcecountry", ""),
            }
        )
    return out


def summarise_articles(articles: list) -> dict:
    """Counts and a few samples from a list of parsed articles."""
    domains = {}
    for a in articles:
        dom = a.get("domain", "")
        if dom:
            domains[dom] = domains.get(dom, 0) + 1
    top_domains = sorted(domains, key=domains.get, reverse=True)[:3]
    titles = [a["title"] for a in articles if a.get("title")][:3]
    return {
        "n_articles": len(articles),
        "has_news": len(articles) > 0,
        "top_domains": "; ".join(top_domains),
        "sample_titles": " || ".join(titles),
    }


class GdeltNews:
    """Polite GDELT Doc client with on-disk caching."""

    # GDELT asks for no more than one request every 5 seconds, so we space calls
    # at least that far apart.
    def __init__(self, cache_dir: str = "data/raw/gdelt_cache", min_interval: float = 5.5):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.min_interval = min_interval
        self._last_call = 0.0
        self._session = None

    def _get_session(self):
        if self._session is None:
            import requests

            self._session = requests.Session()
            self._session.headers.update({"User-Agent": "lloyds-student-project/1.0"})
        return self._session

    def _throttle(self):
        elapsed = time.monotonic() - self._last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self._last_call = time.monotonic()

    def _cache_path(self, key: str) -> Path:
        safe = re.sub(r"[^A-Za-z0-9._-]", "_", key).strip("_")[:120]
        return self.cache_dir / f"{safe}.json"

    def query_articles(self, query: str, timespan: str = "12months",
                       maxrecords: int = 75) -> "dict | None":
        """Call the GDELT Doc API in artlist mode and return parsed JSON.

        Returns None on a failure or an empty/garbled response, so callers can
        treat that as no coverage rather than crashing.
        """
        cache_file = self._cache_path(f"{query}_{timespan}_{maxrecords}")
        if cache_file.exists():
            return json.loads(cache_file.read_text(encoding="utf-8"))

        params = {
            "query": query,
            "mode": "artlist",
            "format": "json",
            "maxrecords": str(maxrecords),
            "timespan": timespan,
            "sort": "datedesc",
        }
        session = self._get_session()
        for attempt in range(4):
            self._throttle()
            try:
                resp = session.get(DOC_API, params=params, timeout=30)
            except Exception:
                time.sleep(6)
                continue
            if resp.status_code == 429:
                # Too fast: wait out the rate limit and retry.
                time.sleep(6)
                continue
            if resp.status_code == 200 and resp.text.strip().startswith("{"):
                try:
                    data = resp.json()
                except ValueError:
                    data = {}
                cache_file.write_text(json.dumps(data), encoding="utf-8")
                return data
            # A 200 with non-JSON text means no results or a soft error.
            if resp.status_code == 200:
                cache_file.write_text("{}", encoding="utf-8")
                return {}
            time.sleep(6)
        return None

    def company_news(self, name: str, timespan: str = "12months",
                     maxrecords: int = 75) -> dict:
        """Get a news summary for one company, searching its cleaned name."""
        cleaned = clean_company_name(name)
        if len(cleaned) < 3:
            # Name too short or generic to search safely.
            return {"query_used": cleaned, "n_articles": 0, "has_news": False,
                    "top_domains": "", "sample_titles": ""}
        query = f'"{cleaned}"'  # quoted phrase to reduce false matches
        payload = self.query_articles(query, timespan, maxrecords)
        summary = summarise_articles(parse_artlist(payload))
        summary["query_used"] = query
        return summary


def main():
    import sys

    name = sys.argv[1] if len(sys.argv) > 1 else "Greggs"
    client = GdeltNews()
    print(json.dumps(client.company_news(name), indent=2))


if __name__ == "__main__":
    main()


## Part 2: The analysis

### Step 1: Load the data and measure how common attrition is

We load the full register, turn the raw fields into signals using the engine above, and measure how common dormancy, distress, and borrowing are, overall and by sector and size. (Loading the full file takes a couple of minutes.)

In [1]:
import pandas as pd

# The full UK company register, downloaded from Companies House
CSV = "../data/raw/BasicCompanyDataAsOneFile-2026-06-01.csv"
WANTED = {"CompanyName", "CompanyNumber", "CompanyCategory", "CompanyStatus",
          "IncorporationDate", "Accounts.AccountCategory", "Accounts.NextDueDate",
          "Mortgages.NumMortCharges", "Mortgages.NumMortOutstanding",
          "Mortgages.NumMortSatisfied", "SICCode.SicText_1", "SICCode.SicText_2",
          "SICCode.SicText_3", "SICCode.SicText_4"}

df = pd.read_csv(CSV, usecols=lambda c: c.strip() in WANTED, dtype=str, low_memory=False)
df.columns = [c.strip() for c in df.columns]
for col in ["Mortgages.NumMortCharges", "Mortgages.NumMortOutstanding", "Mortgages.NumMortSatisfied"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Turn the raw fields into the signals, using the engine functions above
status_map = {v: classify_status(v) for v in df["CompanyStatus"].dropna().unique()}
df["attrition_status"] = df["CompanyStatus"].map(status_map).fillna(STATUS_OTHER)
df["is_distress"] = df["attrition_status"].isin([STRIKE_OFF, INSOLVENCY, DISSOLVED])

acc = df["Accounts.AccountCategory"]
df["is_dormant"] = acc.map({v: classify_accounts_activity(v) for v in acc.dropna().unique()}).eq(DORMANT)
df["size_band"] = acc.map({v: size_band(v) for v in acc.dropna().unique()}).fillna(SIZE_UNKNOWN)

sic1 = df["SICCode.SicText_1"]
df["sector"] = sic1.map({v: target_sector([v]) for v in sic1.dropna().unique()}).fillna(SECTOR_OTHER)
df["has_charges"] = df["Mortgages.NumMortCharges"] > 0

print(f"Companies loaded: {len(df):,}")
print(f"Dormant: {df['is_dormant'].mean():.1%}   "
      f"In distress: {df['is_distress'].mean():.1%}   "
      f"Has a loan: {df['has_charges'].mean():.1%}")
print()
print("Share with a loan, by size band:")
print((df.groupby("size_band")["has_charges"].mean().sort_values(ascending=False) * 100).round(1).to_string())
print()
print("Distress rate, by sector:")
print((df.groupby("sector")["is_distress"].mean().sort_values(ascending=False) * 100).round(1).to_string())

Companies loaded: 5,698,274
Dormant: 11.5%   In distress: 9.2%   Has a loan: 14.3%

Share with a loan, by size band:
size_band
Midcorp    83.2
Large      62.5
SME        27.1
BB         10.9
unknown     5.2

Distress rate, by sector:
sector
Wholesale & retail                      11.5
Other                                   10.8
Manufacturing                           10.2
Fast growth & emerging                   8.3
Technology, legal & professional         7.5
Healthcare                               7.3
Public sector, education & charities     6.7
Agriculture                              5.2
Real estate                              4.1


**What this shows:**
- About 1 in 9 companies is dormant, about 1 in 11 is in some form of distress, and only about 1 in 7 has any loan at all.
- Loans barely exist in the smallest firms and are common in large ones. So any signal based on a company's bank only works for bigger firms.
- Wholesale and retail, then Manufacturing, have the highest distress.

### Step 2: The bank-switch method

When a company borrows, Companies House records the lender's name. By tracking the lender over time we can see a company drop one bank and pick up another. Here is the method on a made-up company that moved from Barclays to Lloyds.

In [1]:
# A made-up company: an old Barclays loan paid off, then a new Lloyds loan taken out
example = parse_charges({"items": [
    {"created_on": "2018-01-01", "satisfied_on": "2022-01-01", "status": "fully-satisfied",
     "persons_entitled": [{"name": "Barclays Bank PLC"}], "classification": {"description": "Charge"}},
    {"created_on": "2022-02-01", "satisfied_on": None, "status": "outstanding",
     "persons_entitled": [{"name": "Lloyds Bank PLC"}], "classification": {"description": "Charge"}},
]})

result = detect_bank_switch(example)
print("Switched bank? ", result["bank_switch"])
print("Lost:  ", result["banks_lost"])
print("Gained:", result["banks_gained"])

Switched bank?  True
Lost:   ['BARCLAYS']
Gained: ['LLOYDS']


Now the same idea on a small live sample of real firms that currently borrow.

In [1]:
# A small live check on real firms that currently have a loan.
# (Small sample so it runs quickly; the full pipeline ran on hundreds of firms.)
sample = df[df["Mortgages.NumMortOutstanding"] > 0].head(40)
client = CompaniesHouseClient(cache_dir="../data/raw/api_cache")

switch = lost_all = with_bank = 0
for num in sample["CompanyNumber"]:
    ch = parse_charges(client.get_charges(str(num).strip()))
    r = detect_bank_switch(ch)
    if r["n_current_banks"] + r["n_past_banks"] > 0:
        with_bank += 1
    switch += int(r["bank_switch"])
    lost_all += int(r["lost_all_banks"])

print(f"Checked {len(sample)} firms that borrow ({with_bank} had a recognised bank):")
print(f"  switched bank (A to B): {switch}")
print(f"  lost all their banks  : {lost_all}")
print()
print("Full pipeline on hundreds of borrowing firms: about 12.5% had switched bank.")

Checked 40 firms that borrow (19 had a recognised bank):
  switched bank (A to B): 2
  lost all their banks  : 2

Full pipeline on hundreds of borrowing firms: about 12.5% had switched bank.


### Step 3: Does losing a bank predict trouble?

We dated the first distress event for each company from its filing history (the code below shows that working), then checked whether losing a bank tends to come first. **It does not.** In the two years before a distress event, only 7.2% of firms had lost a bank, barely above the 5.5% you see anyway.

| Group | Lost a bank in the 24 months before |
| --- | --- |
| Firms that hit distress | 7.2% |
| Firms with no event (baseline) | 5.5% |

So a bank loss is good for describing attrition, not for predicting it.

In [1]:
# Reading a company's filing history to date its first sign of trouble
filings = parse_filing_history({"items": [
    {"date": "2023-05-01", "category": "accounts", "type": "AA",
     "description": "accounts-with-accounts-type-dormant"},
    {"date": "2024-02-01", "category": "gazette", "type": "GAZ1",
     "description": "first-gazette-notice-for-compulsory-strike-off"},
]})
events = extract_attrition_events(filings)
print("First distress event:", events["first_event_type"], "on", events["first_event_date"])

First distress event: dormant_accounts on 2023-05-01


### Step 4: The news check

Finally, does news coverage help? We check a few companies live. A large firm has plenty of coverage; small firms have none. Across a sample of 100 small firms earlier, not one had any news. So news only helps for larger, well known companies.

In [1]:
# Does news coverage exist for these companies? (live, a few calls)
news = GdeltNews(cache_dir="../data/raw/gdelt_cache")
for name in ["Greggs PLC", "JMC LIFTS LTD", "FINASOFT LIMITED"]:
    n = news.company_news(name)["n_articles"]
    print(f"{name:20s} -> {n} news articles in the last year")

Greggs PLC           -> 21 news articles in the last year
JMC LIFTS LTD        -> 0 news articles in the last year
FINASOFT LIMITED     -> 0 news articles in the last year


## What this all means

- The one signal that clearly works is the **bank switch**: spotting a healthy company moving its borrowing to a rival.
- **Losing a bank does not predict** a company failing.
- For small firms, loans and news are mostly absent, so the reliable warning signs are the simple Companies House ones: **late filings, going dormant, and status changes**.
- The bank and news signals are better aimed at **larger firms**.

In short: for the small-firm scope, watch the simple public filings. For the bank and news ideas to pay off, the project would need to look at larger firms.